In [ ]:
import asyncio
import matplotlib.pyplot as plt

import cv2
import numpy as np
from pyproj import Transformer

from uavcalibration.map import *


wgs2merc = Transformer.from_crs("epsg:4326", "epsg:3857", always_xy=True)


def lonlat2tile(xx: np.ndarray, yy: np.ndarray, zoom: int = 0):
    merc = np.array(wgs2merc.transform(xx, yy))
    # reverse y axie
    merc[1] *= -1
    # normalize to [0,1]
    merc /= 2 * 20037508.342789244
    merc += 0.5

    map_size = 1 << zoom
    global_xy = merc * map_size
    tile_xy = (global_xy).astype(np.uint32)
    pixel_xy = (global_xy % 1 * 256).astype(np.uint32)

    return tile_xy, pixel_xy


bbox = (119.805926, 32.355491, 119.815926, 32.345491)
# bbox = (119.805926, 32.355491, 119.809926, 32.352491)
zoom = 18
wgs = np.array([*bbox, 180, 85.051129, -180, -85.051129]).reshape(-1, 2).T
lonlat2tile(*wgs, zoom=zoom)

In [ ]:
tiled_map = TiledMap(
    "https://webst01.is.autonavi.com/appmaptile?style=6&x={x}&y={y}&z={z}",
)
merc_xy = np.array(wgs2merc.transform(*wgs))
merc_xy

In [ ]:
global_xy = tiled_map.merc2tile(*merc_xy, zoom=zoom).astype(np.uint64)
tile_xy, pixel_xy = np.divmod(global_xy, tiled_map.tile_size)
(tile_xy, pixel_xy)

In [ ]:
async with tiled_map:
    image, trans = await tiled_map.get_async(bbox)
plt.imshow(image)
plt.axis("off")
image.shape[:2]

In [ ]:
coords = np.array([0, 0, *image.shape[:2]], np.float64).reshape(-1, 2)
trans.apply(coords)